# `Занятие 9: Transformers + 🤗`

# `0. Решение реальных задач`

Давайте теперь попробуем решить какую-нибудь задачу с помощью трансформеров. На прошлом семинаре мы разобрали seq-to-seq задачу машинного перевода, а теперь попробуем решить пару других задач.

Очень популярной задачей является классификация текста. Давайте вспомним, как она решается в общем случае. Разобьем задачу на этапы:

0. Собираем где-то данные или берем уже размеченный датасет.
1. Токенизируем текст.
2. Получаем признаки.
2. Обучаем классификатор на этих признаках.

На каждом из этапов у нас могут быть проблемы:
0. У нас может быть мало данных для обучения.
1. Корпус может быть очень большим и матрица эмбеддингов будет очень объемной.
2. OOV.
2. Классификатор может не выучивать какие-то сложные зависимости в тексте.

Исправить ситуацию в 0 пункте напрямую очень сложно и дорого. Для 1 и 2 пункта мы сейчас разберем способ кодирования, который позволит решить эти проблемы. 3 пункт решается усложнением семейства моделей (в данном случае, обогащением с помощью attention)

# `1. Токенизация`

Очевидно, что мы не можем засунуть текст в модель – она ожидает получить вектор.
В трансформерах используется обучаемая матрица эмбеддингов, но нам все равно нужно перевести изначальное слово в OHE. И тут возникают проблемы:

1. В богатых на словоформы языках типа русского слов с одним корнем может быть очень много слов -> гигантская размерность словаря -> гигантская размерность обучаемой матрицы эмбеддингов.
2. Мы всегда встретим какое-то новое слово, являющееся формой или комбинацией уже известных, как например в немецком.

Эти проблемы помогает решить особый способ токенизации. Ниже разберем самый популярный вариант, который использовался в трансформере, а пока давайте посмотрим на иллюстрацию, которая помогает понять смысл токенизации:


<img src="https://www.oreilly.com/api/v2/epubs/9781492062561/files/assets/anlp_0401.png" width="700">

[источник картинки](https://www.oreilly.com/api/v2/epubs/9781492062561/files/assets/anlp_0401.png)

**BPE**

Byte-Pair Encoding (BPE) изначально разрабатывался как алгоритм для сжатия текстов, а затем использовался Google и OpenAI для токенизации при предварительном обучении моделей Transformer и GPT. Он также используется многими Transformer моделями, включая GPT, GPT-2, RoBERTa, BART и DeBERTa.

Обучение BPE начинается с вычисления уникального набора слов, используемых в корпусе (после завершения шагов нормализации и предварительной токенизации), а затем построения словаря путем использования всех символов, используемых для написания этих слов. В качестве очень простого примера предположим, что наш корпус использует эти пять слов:

```
"hug", "pug", "pun", "bun", "hugs"
```

Тогда базовый словарь будет `["b", "g", "h", "n", "p", "s", "u"]`. Для реальных случаев этот базовый словарь будет содержать как минимум все символы ASCII и, возможно, также некоторые символы Unicode. Если пример, который вы токенизируете, использует символ, которого нет в учебном корпусе, этот символ будет преобразован в неизвестный токен. Это одна из причин, почему многие модели NLP очень плохо анализируют контент, например, со смайликами.


> У токенизаторов GPT-2 и RoBERTa (которые очень похожи) есть хитрый способ справиться с этим: они рассматривают слова как написанные не символами Unicode, а байтами. Таким образом, базовый словарь имеет небольшой размер (256), но каждый символ, который вы можете придумать, все равно будет включен и не будет преобразован в неизвестный токен. Этот трюк называется byte-level BPE.

После получения этого базового словаря мы добавляем новые токены до тех пор, пока не будет достигнут желаемый размер словаря путем обучения слияниям, которые представляют собой правила объединения двух элементов существующего словаря вместе в новый. Итак, вначале эти слияния будут создавать токены с двумя символами, а затем, по мере обучения, более длинные подслова.

На любом этапе обучения токенизатора алгоритм BPE будет искать наиболее часто встречающуюся пару существующих токенов (под «парой» здесь мы подразумеваем два последовательных токена в слове). Эта наиболее часто встречающаяся пара будет объединена, мы добавим ее в словарь и повторим процедуру.

Возвращаясь к нашему предыдущему примеру, давайте предположим, что слова имели следующие частоты:
```
("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)
```
означает, что "hug" присутствовало в корпусе 10 раз, "pug" 5 раз, "pun" 12 раз, "bun" 4 раза и "hugs" 5 раз. Мы начинаем обучение, разбивая каждое слово на символы (те, которые формируют наш первоначальный словарный запас), чтобы мы могли видеть каждое слово как список токенов:
```
("h" "u" "g", 10), ("p" "u" "g", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "u" "g" "s", 5)
```

Затем мы рассматриваем пары. Пара ("h", "u") присутствует в словах "hug" и "hugs", всего 15 раз в корпусе. Однако это не самая часто встречающаяся пара: ("u", "g") присутствует в "hug", "pug" и "hugs" 20 раз в словаре.

Таким образом, первое правило слияния, изученное токенизатором, это ("u", "g") -> "ug", что означает, что "ug" будет добавлено в словарь, и пара должна быть объединена во всех словах корпуса. В конце этого этапа словарь и корпус выглядят следующим образом:
```
Vocabulary: ["b", "g", "h", "n", "p", "s", "u", "ug"]
Corpus: ("h" "ug", 10), ("p" "ug", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "ug" "s", 5)
```

Теперь у нас есть несколько пар, результатом которых является токен длиннее двух символов: например, пара ("h", "ug") (присутствует в корпусе 15 раз). Однако наиболее часто встречающаяся пара на этом этапе — («u», «n») — присутствует в корпусе 16 раз, поэтому второе усвоенное правило слияния — («u», «n») -> «un». Добавление этого в словарь и объединение всех существующих вхождений приводит нас к следующему:
```
Vocabulary: ["b", "g", "h", "n", "p", "s", "u", "ug", "un"]
Corpus: ("h" "ug", 10), ("p" "ug", 5), ("p" "un", 12), ("b" "un", 4), ("h" "ug" "s", 5)
```

Теперь наиболее часто встречающаяся пара — («h», «ug»), поэтому мы изучаем правило слияния («h», «ug») -> «hug», которое дает нам наш первый трехбуквенный токен. После слияния корпус выглядит так:
```
Vocabulary: ["b", "g", "h", "n", "p", "s", "u", "ug", "un", "hug"]
Corpus: ("hug", 10), ("p" "ug", 5), ("p" "un", 12), ("b" "un", 4), ("hug" "s", 5)
```
И так продолжаем до тех пор, пока не достигнем желаемого размера словаря.

Что происходит после того, как мы изучили правила слияния и попробовали применить их к новому тексту:

1. Нормализация
2. Предварительная токенизация
3. Разделение слов на отдельные символы
4. Применение изученных правил слияния по порядку к этим разбиениям

Давайте возьмем пример, который мы использовали во время обучения, с тремя изученными правилами слияния:
```
("u", "g") -> "ug"
("u", "n") -> "un"
("h", "ug") -> "hug"
```

Слово "bug" будет обозначено как `["b", "ug"]`. "mug", однако, будет обозначаться как `["[UNK]", "ug"]`, так как буква "m" не входила в базовый словарь. Точно так же слово "thug" будет обозначено как `["[UNK]", "hug"]`: буква "t" отсутствует в базовом словаре, и применение правил слияния приводит сначала к слиянию "u" и "g", а затем к слиянию "hu" и "g".



**WordPiece**

Как и BPE, WordPiece начинается с небольшого словаря, включающего специальные токены, используемые моделью, и исходный алфавит. Поскольку он идентифицирует подслова, добавляя префикс (например, ## для BERT), каждое слово изначально разбивается путем добавления этого префикса ко всем символам внутри слова. Так, например, "word" разбивается следующим образом:
```
w ##o ##r ##d
```
Таким образом, исходный алфавит содержит все символы, присутствующие в начале слова, и символы, присутствующие внутри слова, которому предшествует префикс WordPiece.

Затем, опять же, как BPE, WordPiece изучает правила слияния. Основное отличие состоит в том, как выбирается объединяемая пара. Вместо того, чтобы выбирать наиболее часто встречающуюся пару, WordPiece вычисляет оценку для каждой пары, используя следующую формулу:
$$score=\frac{\text{freq_of_pair}}{\text{freq_of_first_element}\times\text{freq_of_second_element}}$$

Разделив частоту пары на произведение частот каждой из ее частей, алгоритм отдает приоритет слиянию пар, в которых отдельные части реже встречаются в словаре. Например, оно не обязательно будет объединять ("un", "##able"), даже если эта пара очень часто встречается в словаре, потому что две пары "un" и "##able", скорее всего, будут часто встречаться в разных словах. Напротив, пара вроде ("hu", "##gging"), вероятно, будет сливаться быстрее (при условии, что слово "hugging" часто встречается в лексиконе), поскольку "hu" и "##gging", скорее всего, будут реже встречаться индивидуально.

Давайте посмотрим на тот же словарь, который мы использовали в примере обучения BPE:
```
("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)
```
Разбиение будет следующим:
```
("h" "##u" "##g", 10), ("p" "##u" "##g", 5), ("p" "##u" "##n", 12), ("b" "##u" "##n", 4), ("h" "##u" "##g" "##s", 5)
```




поэтому исходный словарь будет `["b", "h", "p", "##g", "##n", "##s", "##u"]` (если забыть о специальных токенах на данный момент). Наиболее часто встречается пара ("##u", "##g") (присутствует 20 раз), но индивидуальная частота "##u" очень высока, поэтому его оценка не самая высокая (это 1/36). Все пары с "##u" на самом деле имеют одинаковую оценку (1/36), поэтому лучший результат получает пара ("##g", "##s") — единственная без "##u" — в 1/20, и первое изученное слияние это ("##g", "##s") -> ("##gs").

И так продолжаем до тех пор, пока не достигнем желаемого размера словаря.

# `2. BERT`

**Какие пайплайны для классификации текстов мы уже знаем?**

Как мы обсудили на предыдущем семинаре, архитектура Transformer существенно увеличила качество на задачах seq-to-seq, однако, еще большее влияние на сферу NLP она произвела благодаря механизму self-attention, который позволил трансформерам отлично понимать текст.

Нам хотелось бы научиться извлекать признаки из текста, агрегируя в векторы токенов их контекст. Давайте рассмотрим модель **BERT** (Bidirectional Encoder Representations from Transformers) [cite](https://arxiv.org/pdf/1810.04805.pdf).

Механизм self-attention, использующийся в трансформере позволяет довольно хорошо вытаскивать из текста информацию о контексте. Логично ожидать, что эмбеддинги слов, получающиеся на выходе энкодера очень хорошо агрегируют в себе контекст. Эта идея лежит в основе архитектуры BERT:

<center>
<div>
<img src="https://jalammar.github.io/images/bert-base-bert-large-encoders.png", width="700">
</div>
</center>

Параллельно с хорошими контекстными представлениями, BERT решает еще одну важную задачу – недостаток данных для обучения (w2v тоже умеет ее решать).

**Masked Language Modeling** (Pre-training)

BERT обучался на большом корпусе текстов предсказывать скрытые токены. Как раз для этого в механизме self-attention на уровне кода предусмотрено маскирование (см. предыдущий семинар)

<center>
<div>
<img src="https://jalammar.github.io/images/BERT-language-modeling-masked-lm.png", width="700">
</div>
</center>

**Next Sentence Prediction** (Pre-training)

Дополнительной задачей была так называемая NSP (Next Sentence prediction)

<center>
<div>
<img src="https://jalammar.github.io/images/bert-next-sentence-prediction.png", width="700">
</div>
</center>

Стоит отметить, что от решения этой задачи в дальнейшем отказались (например в roBERTa)

Отдельно посмотрим, как в BERT получаются эмбеддинги токенов:

<img src="https://sun9-77.userapi.com/impg/VmvOBfpLdaCMLOdIA3ptZNRwp1-FuRMDpsCFNw/umfQOckVOQo.jpg?size=2560x753&quality=96&sign=48a3251e73127125a9f2d5be51ed6131&type=album" width="700">

**Transfer learning** (Fine-tunning)

Вообще говоря, BERT не решает какую-то конечную задачу сам по себе, он используется как предобученные контекстные признаки текста, которые потом "дообучаются" уже на конкретную задачу. Далее в нашем ноутбуке мы разберем конкретный пример.

<center>
<div>
<img src="https://sun9-4.userapi.com/impg/5GtCOYwymarRQjiXVxpX6Lz1epl-BAyyo1oyJg/yz60gLfm5Io.jpg?size=2221x2160&quality=96&sign=a29e275e42d346cf6613a2dbd7cfbe6b&type=album", width="1000">
</div>
</center>

BERT стал базой для очень многих архитектур, вот самые популярные:
- [RoBERTa](https://arxiv.org/pdf/1907.11692.pdf)
- [ALBERT](https://arxiv.org/pdf/1909.11942.pdf)
- [DeBERT](https://arxiv.org/pdf/2006.03654.pdf)
- [DistilBERT](https://arxiv.org/pdf/1910.01108.pdf)
- ...

**Материалы**
1. https://jalammar.github.io/illustrated-bert/


# `3. 🤗Transformers`

Итак, мы разобрались с проблемами, которые описаны выше, давайте теперь посмотрим как можно использовать BERT для решения конечной задачи.

## `3.1 HuggingFace`

К статье Attention is all you need был приложен код трансформера, написанный на TensorFlow. Мы также посмотрели примеры функций, которые позволяют реализовать Transformer с нуля самостоятельно, однако, куча готовых библиотек уже написаны для этих целей. Самая популярная из них – HuggingFace. Она поддерживает различные DL фреймворки, в частости – PyTorch. 

Плюсы HF:
- Куча готовых моделей
- Интеграция с PyTorch / TensorFlow
- Очень много готовых датасетов
- **Большой HUB с предобученными весами**

In [ ]:
# @markdown Скачаем HuggingFace
from IPython.display import clear_output
!pip install transformers datasets evaluate
clear_output() # Если вы хотите почистить вывод ячейки не руками

In [ ]:
# @markdown Импортируем важные вещи
import torch
from torch import nn
import numpy as np

**Pipeline**

Pipeline — это самый простой и быстрый способ использовать предварительно обученную модель. Вы можете использовать `pipeline()` из коробки для многих задач в разных модальностях, некоторые из которых показаны в таблице ниже:

| **Task**                     | **Description**                                                                                              | **Modality**    | **Pipeline identifier**                       |
|------------------------------|--------------------------------------------------------------------------------------------------------------|-----------------|-----------------------------------------------|
| Text classification          | assign a label to a given sequence of text                                                                   | NLP             | pipeline(task=“sentiment-analysis”)           |
| Text generation              | generate text given a prompt                                                                                 | NLP             | pipeline(task=“text-generation”)              |
| Summarization                | generate a summary of a sequence of text or document                                                         | NLP             | pipeline(task=“summarization”)                |
| Image classification         | assign a label to an image                                                                                   | Computer vision | pipeline(task=“image-classification”)         |
| Image segmentation           | assign a label to each individual pixel of an image (supports semantic, panoptic, and instance segmentation) | Computer vision | pipeline(task=“image-segmentation”)           |
| Object detection             | predict the bounding boxes and classes of objects in an image                                                | Computer vision | pipeline(task=“object-detection”)             |
| Audio classification         | assign a label to some audio data                                                                            | Audio           | pipeline(task=“audio-classification”)         |
| Automatic speech recognition | transcribe speech into text                                                                                  | Audio           | pipeline(task=“automatic-speech-recognition”) |
| Visual question answering    | answer a question about the image, given an image and a question                                             | Multimodal      | pipeline(task=“vqa”)                          |
| Document question answering  | answer a question about a document, given an image and a question                                            | Multimodal      | pipeline(task="document-question-answering")  |
| Image captioning             | generate a caption for a given image                                                                         | Multimodal      | pipeline(task="image-to-text")                |

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


`pipeline()` загружает и кэширует предварительно обученную модель по умолчанию и токенизатор для нашей конкретной задачи:

In [ ]:
classifier("We are very happy to show you the 🤗 Transformers library.")

[{'label': 'POSITIVE', 'score': 0.9997795224189758}]

Мы можем подавать в него сразу несколько предложений или даже полноценный датасет:

In [ ]:
results = classifier(["We are very happy to show you the 🤗 Transformers library.", "We hope you don't hate it."])
for result in results:
    print(f"label: {result['label']}, with score: {round(result['score'], 4)}")

label: POSITIVE, with score: 0.9998
label: NEGATIVE, with score: 0.5309


Однако, такой пайплайн не получится дообучить на конкретную задачу. Давайте спустимся на уровень ниже и пройдемся по всем этапам NLP в HF Transformers по порядку.

## `3.2 🤗Datasets`

Чтобы работать с текстом, нам нужно создать из него датасет.
https://huggingface.co/docs/datasets/tutorial

### Готовые датасеты
В HuggingFace есть большое количество готовых данных под разные задачи. Давайте скачаем датасет `rotten_tomatoes`, содержащий отзывы на фильмы.

In [ ]:
from datasets import load_dataset
rotten_tomatoes = load_dataset("rotten_tomatoes")
clear_output()

Посмотрим, что содержится в этом датасете

In [ ]:
rotten_tomatoes

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [ ]:
rotten_tomatoes['train'][0]

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
 'label': 1}

### Кастомный датасет
Конечно, есть довольно простой интерфейс для загрузки собственных данных

In [ ]:
from datasets import Dataset
ds = Dataset.from_dict({"pokemon": ["bulbasaur", "squirtle"], "type": ["grass", "water"]})
ds[0]

In [ ]:
data_files = {"train": "train.csv", "test": "test.csv"}
dataset = load_dataset("namespace/your_dataset_name", data_files=data_files)

## `3.3 Подготовка данных, 🤗Tokenizers`


Прежде чем вы сможете обучить модель на наборе данных, ее необходимо предварительно обработать до ожидаемого входного формата модели. Независимо от того, являются ли ваши данные текстом, изображениями или звуком, их необходимо преобразовать и собрать в пакеты тензоров. 🤗Transformers предоставляет набор классов предварительной обработки, помогающих подготовить данные для модели.

Основным инструментом предварительной обработки текстовых данных является **токенизатор**. Токенизатор разбивает текст на токены в соответствии с набором правил. Токены преобразуются в числа, а затем в тензоры, которые становятся входными данными модели. Любые дополнительные входные данные, требуемые моделью, добавляются токенизатором.

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

clear_output()

Отправим что-нибудь в наш токенизатор:

In [ ]:
encoded_input = tokenizer("Do not meddle in the affairs of wizards, for they are subtle and quick to anger.")
encoded_input

{'input_ids': [101, 2091, 1136, 1143, 13002, 1107, 1103, 5707, 1104, 16678, 1116, 117, 1111, 1152, 1132, 11515, 1105, 3613, 1106, 4470, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Токенизатор возвращает словарь с тремя важными элементами:
- `input_ids` — это индексы, соответствующие каждому токену в предложении.
- `attention_mask` указывает, следует ли обращать внимание на токен или нет.
- `token_type_ids` указывает, к какой последовательности принадлежит токен, если существует более одной последовательности.

Вернем наш ввод, расшифровав `input_ids`:

In [ ]:
tokenizer.decode(encoded_input["input_ids"])

'[CLS] Do not meddle in the affairs of wizards, for they are subtle and quick to anger. [SEP]'

Как видите, токенизатор добавил в предложение два специальных токена — CLS и SEP (классификатор и разделитель). Не всем моделям нужны специальные токены, но если они нужны, токенизатор автоматически добавит их за вас.

Если есть несколько предложений, которые вы хотите предварительно обработать, передайте их в виде списка токенизатору:

In [ ]:
batch_sentences = [
    "But what about second breakfast?",
    "Don't think he knows about second breakfast, Pip.",
    "What about elevensies?",
]
encoded_inputs = tokenizer(batch_sentences)
encoded_inputs['input_ids']

[[101, 1252, 1184, 1164, 1248, 6462, 136, 102],
 [101,
  1790,
  112,
  189,
  1341,
  1119,
  3520,
  1164,
  1248,
  6462,
  117,
  21902,
  1643,
  119,
  102],
 [101, 1327, 1164, 5450, 23434, 136, 102]]

**Padding**

Предложения не всегда имеют одинаковую длину, что может быть проблемой, потому что тензоры, входные данные модели, должны иметь единую форму. Padding — это стратегия обеспечения прямоугольности тензоров путем добавления специального pad-токена к более коротким предложениям.

In [ ]:
batch_sentences = [
    "But what about second breakfast?",
    "Don't think he knows about second breakfast, Pip.",
    "What about elevensies?",
]
encoded_input = tokenizer(batch_sentences, padding=True)
encoded_input['input_ids']

[[101, 1252, 1184, 1164, 1248, 6462, 136, 102, 0, 0, 0, 0, 0, 0, 0],
 [101,
  1790,
  112,
  189,
  1341,
  1119,
  3520,
  1164,
  1248,
  6462,
  117,
  21902,
  1643,
  119,
  102],
 [101, 1327, 1164, 5450, 23434, 136, 102, 0, 0, 0, 0, 0, 0, 0, 0]]

Первое и третье предложения теперь дополнены нулями, потому что они короче.

**Truncation**

С другой стороны, иногда последовательность может быть слишком длинной для модели. В этом случае нужно укоротить последовательность до более короткой длины.

Установите для параметра `truncation` значение `True`, чтобы обрезать последовательность до максимальной длины, приемлемой для модели:

In [ ]:
batch_sentences = [
    "But what about second breakfast?",
    "Don't think he knows about second breakfast, Pip.",
    "What about elevensies?",
]
encoded_input = tokenizer(batch_sentences, padding=True, truncation=True, max_length=5) # just for example
encoded_input['input_ids']

[[101, 1252, 1184, 1164, 102],
 [101, 1790, 112, 189, 102],
 [101, 1327, 1164, 5450, 102]]

Наконец, если нужно, чтобы токенизатор возвращал фактические тензоры, которые передаются модели, установите для параметра `return_tensors` значение `pt` для `PyTorch` (или `tf` для `TensorFlow`):

In [ ]:
batch_sentences = [
    "But what about second breakfast?",
    "Don't think he knows about second breakfast, Pip.",
    "What about elevensies?",
]
encoded_input = tokenizer(batch_sentences, padding=True, truncation=True, return_tensors="pt")
encoded_input['input_ids']

tensor([[  101,  1252,  1184,  1164,  1248,  6462,   136,   102,     0,     0,
             0,     0,     0,     0,     0],
        [  101,  1790,   112,   189,  1341,  1119,  3520,  1164,  1248,  6462,
           117, 21902,  1643,   119,   102],
        [  101,  1327,  1164,  5450, 23434,   136,   102,     0,     0,     0,
             0,     0,     0,     0,     0]])


Почитать подробнее про обработку звука и картинок можно тут [cite](https://github.com/huggingface/notebooks/blob/main/transformers_doc/en/preprocessing.ipynb).

Вам вряд ли придется дообучать токенизатор, или даже обучать его с нуля, так как HF HUB содержит достаточное количество токенизаторов и моделей под любые задачи. Но если вдруг вам это понадобится – можно почитать [тут](https://huggingface.co/course/chapter6/2)

## `3.4 Создание моделей`

Чтобы кастомизировать `pipeline()` или просто загрузить какую-то конкретную модель, можно воспользоваться классом `AutoClass`

Классы `AutoModelForSequenceClassification` и `AutoTokenizer` работают вместе, чтобы обеспечить работу `pipeline()` или `Trainer()` (обсудим далее). Автокласс — это ярлык, который автоматически извлекает архитектуру предварительно обученной модели по ее имени или пути. Вам нужно только выбрать соответствующий автокласс для вашей задачи.

🤗Transformers предоставляет простой и унифицированный способ загрузки предварительно обученных моделей. Это означает, что вы можете загрузить `AutoModel`, как если бы вы загружали `AutoTokenizer`, используя метод `from_pretrained`. Единственная разница заключается в выборе правильной `AutoModel` для задачи. Для классификации текста (или последовательности) вы должны загрузить `AutoModelForSequenceClassification`:

In [ ]:
from transformers import AutoModelForSequenceClassification


# Загружать модели из HUB-а нужно только из проверенных источников!!!
MODEL_NAME = "blanchefort/rubert-base-cased-sentiment"
pt_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Давайте посмотрим из чего состоит экземпляр класса `AutoModelForSequenceClassification` 

In [ ]:
type(pt_model)

transformers.models.bert.modeling_bert.BertForSequenceClassification

In [ ]:
pt_model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

Получим предсказания для нескольких текстов:

In [ ]:
texts = ["Я очень рад проводить семинар по работе с 🤗Трансформерс.", 
         "Это был отвратительный сериал, хотелось бы вернуть свои 10 долларов за подписку.",
         "Завтра ожидается +10 градусов, солнечно."]

pt_batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt",
)

In [ ]:
pt_outputs = pt_model(**pt_batch)
pt_outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-1.0970,  2.1079, -2.4258],
        [-0.0597, -1.0394,  1.3657],
        [ 1.8087,  0.2085, -1.3981]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [ ]:
from torch import nn

pt_predictions = nn.functional.softmax(pt_outputs.logits, dim=-1)
print(pt_predictions)

tensor([[0.0386, 0.9512, 0.0102],
        [0.1807, 0.0678, 0.7515],
        [0.8049, 0.1625, 0.0326]], grad_fn=<SoftmaxBackward0>)


Или сразу:

In [ ]:
classifier = pipeline("sentiment-analysis", model=pt_model, tokenizer=tokenizer)
classifier(texts)

[{'label': 'POSITIVE', 'score': 0.9511977434158325},
 {'label': 'NEGATIVE', 'score': 0.7514975666999817},
 {'label': 'NEUTRAL', 'score': 0.8049389719963074}]

Вот несколько моделей, которые можно загрузить из HF: `ai-forever/ruRoberta-large`, `gpt2`, `t5-base`, `openai/whisper-large` \\
Полная библиотека: https://huggingface.co/models

**Не забывать про безопасность загрузки из HUB-a**

Естественно, модели можно дообучать под конкретные задачи, давайте разберемся с этим.

## `3.5 Fine-tunning моделей, Trainer`

Этап обучения BERT, на котором он учится понимать текст, называется pre-training, мы обсудили его выше. Когда мы хотим обучить предобученный BERT для решения конкретной задачи, то называем этот процесс fine-tunning.
Давайте теперь попробуем дообучиться на конкретной задаче:

### Trainer

Все модели представляют собой объект класса, наследуемый от `torch.nn.Module`, поэтому их можно использовать в любом обычном тренировочном цикле, однако, 🤗Transformers предоставляет класс `Trainer` для `PyTorch`, который содержит базовый цикл обучения и добавляет дополнительный функционал для таких вещей, как распределенное обучение, mixed precision и многое другое.

В зависимости от вашей задачи вы обычно передаете в `Trainer` следующие параметры:

1. [PreTrainedModel](https://huggingface.co/docs/transformers/main/en/main_classes/model#transformers.PreTrainedModel) или [`torch.nn.Module`](https://pytorch.org/docs/stable/nn.html#torch.nn.Module):

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased")

Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertForSequenceClassification: ['vocab_layer_norm.bias', 'vocab_transform.bias', 'vocab_transform.weight', 'vocab_projector.weight', 'vocab_projector.bias', 'vocab_layer_norm.weight']
- This IS expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.weight', 'pre_classifier.bias', 'pre_classi

2. [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments) содержит гиперпараметры модели, которые вы можете изменить, такие как скорость обучения, размер пакета и количество эпох для обучения. Значения по умолчанию используются, если вы не укажете какие-либо обучающие аргументы:

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="path/to/save/folder/",
    logging_strategy="steps",
    evaluation_strategy="steps",
    optim="adamw_torch",
    logging_steps=250,
    disable_tqdm=False,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
)

3. Класс предварительной обработки, такой как tokenizer, image processor, feature extractor, или processor:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

4. Создаем датасет:

In [ ]:
from datasets import load_dataset

dataset = load_dataset("rotten_tomatoes")

  0%|          | 0/3 [00:00<?, ?it/s]

5. Создаем функцию для токенизации датасета:

In [ ]:
def tokenize_dataset(dataset):
    return tokenizer(dataset["text"])

И применяем ее к нашим данным, используя функцию [map](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.map):

In [ ]:
dataset = dataset.map(tokenize_dataset, batched=True)

6. [DataCollatorWithPadding](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DataCollatorWithPadding), чтобы создать батчи из нашего датасета:

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

7. Загрузим метрику, чтобы измерять качество на каждой эпохе:

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

Собираем теперь все вместе в [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer):

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

И запускаем процесс обучения:

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
250,0.095600,0.818684,0.821764
500,0.056900,0.776920,0.833021
750,0.027100,0.813054,0.847092
1000,0.025300,0.906190,0.833959
1250,0.020000,0.899133,0.838649


TrainOutput(global_step=1335, training_loss=0.04327240400992976, metrics={'train_runtime': 214.5059, 'train_samples_per_second': 198.829, 'train_steps_per_second': 6.224, 'total_flos': 579351859980552.0, 'train_loss': 0.04327240400992976, 'epoch': 5.0})

### Хитрости обучения

Я кратко пройдусь по некоторым опциям, которые могут быть полезны:

0. Warming up
1. Weights freezing
2. Mixed precision training
3. Gradient accumulating
4. Optimizers (e.g. 8-bit Adam)

**Warming up**

<img src="https://nlp.seas.harvard.edu/images/the-annotated-transformer_69_0.png" widtg="200">

Этот скорее эвристический метод используется для изменения learning rate во время обучения, добавляется просто параметрами `Trainer`

In [ ]:
additional_training_args = {
    "warmup_steps": 200,
    "max_steps": 700,
}

**Заморозка весов**

В некоторых случаях нам может понадобиться заморозка весов:

In [ ]:
for param in model.distilbert.parameters():
    param.requires_grad = False

**Mixed precision**

Наша модель обучается с помощью float32 градиентов. Возможно, эта точность излишняя, давайте попробуем обучать используя float16. Очень часто, этой точности хватает для тренировки, при этом мы получаем существенное ускорение процесса обучения:

In [ ]:
additional_training_args = {
    "fp16": True,
}

**Gradient accumulating**

Часто, желаемый размер батча просто не помещается в нашу видеокарту. Давайте просто делить батч на несколько подбатчей и аккумулировать результаты:

In [ ]:
additional_training_args = {
    'per_device_train_batch_size': 12,
    'per_device_eval_batch_size': 12,
    'gradient_accumulation_steps': 2,
    'gradient_checkpointing': True,
}

**Кастомный Trainer**

Вам по каким-то причинам может понадобиться переписать класс `Trainer`, например, для того, чтобы ввести веса в функции потерь:

In [ ]:
class MyTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        # forward pass
        weights = torch.ones(2)
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # compute custom loss (suppose one has 3 labels with different weights)
        loss_fct = nn.CrossEntropyLoss(weight=weights.to(device))
        # print(logits.device, labels.device)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

### ✨PyTorch training loop✨


Если по каким-то причинам вы желаете обучаться традиционным способом – you are welcome!
[Подробнее](https://huggingface.co/docs/transformers/training#train-in-native-pytorch)

Чтобы освободить памать GPU можно перезагрузить ноутбук или выполнить следующий код:

In [ ]:
del model
del trainer
torch.cuda.empty_cache()

Класс `Trainer` выполнял кучу грязной работы за нас – теперь придется делать это руками. Плюс, некоторые изменения:
1. Удалим колонку `"text"` – она не нужна для обучения
2. Переименуем `"label"` в `"labels"`
3. Установим нужный формат датасета

In [ ]:
tokenized_datasets = dataset.map(tokenize_dataset)

tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

`Trainer` сам разбивал датасет на батчи, теперь нам придется сделать это самим:

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(tokenized_datasets, shuffle=True, batch_size=8)
eval_dataloader = DataLoader(tokenized_datasets, batch_size=8)

Создадим нашу модель еще раз:

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=3)

Будем использовать `AdamW` оптимизатор:

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

Будем изменять тем обучения по ходу:

In [ ]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

Перенесем наши вычисления на видеокарту:

In [ ]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

Хорошо знакомый:

In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

## `3.6` Adapters


При fine-tunning мы дообучаем все параметры моделей. Есть альтернативный и весьма эффективный подход [cite](https://arxiv.org/pdf/1902.00751.pdf)
<center>
<img src="https://sun9-24.userapi.com/impg/1oNo7gjbJoA2B2rPkr8ASxaCPnJ_ziGTSP1mbw/XtQc0lcgwsk.jpg?size=1532x1140&quality=96&sign=2f18bcb8bf6f49847ff4974c4aa03f59&type=album" width="700">
</center>

In [ ]:
from transformers import AutoModelWithHeads

model = AutoModelWithHeads.from_pretrained("bert-base-uncased")
adapter_name = model.load_adapter("AdapterHub/bert-base-uncased-pf-imdb", source="hf")
model.active_adapters = adapter_name

In [ ]:
from transformers import list_adapters

# source can be "ah" (AdapterHub), "hf" (hf.co) or None (for both, default)
adapter_infos = list_adapters(source="hf", model_name="bert-base-uncased")

Подробнее про использование адаптеров можно почитать на сайте HF [cite](https://huggingface.co/docs/hub/adapter-transformers)

## `3.7 LoRA`

Еще одним подходом, который ускоряет fine-tunning, является Low-Rank Adaptation

<img src="https://adapterhub.ml/static/images/lora.png" width="400">

Low-Rank Adaptation (LoRA) — это эффективный метод дообучения, предложенный [Hu et al. (2021)](https://arxiv.org/pdf/2106.09685.pdf). LoRA вводит обучаемые матрицы разложения низкого ранга в слои предварительно обученной модели. Поэтому для любого слоя модели, выраженного в виде умножения матриц формы $h=W_0x$, он выполняет перепараметризацию, так что:
$$
h = W_0x + \frac{\alpha}{r}BAx
$$
где $A \in R^{r \times k}$ $B \in R^{d \times r}$ – являются матрицами разложения, а $r$ – низкоразмерный ранг разложения, является наиболее важным гиперпараметром.

Хотя, в принципе, эта репараметризация может быть применена к любой матрице весов в модели, исходная статья адаптирует только веса self-attention. `adapter-transformers` дополнительно позволяют внедрять LoRA в FCN слои в промежуточных и выходных компонентах блока Transformer. Вы можете настроить места, в которые должны быть добавлены веса LoRA, используя атрибуты в классе `LoRAConfig`.

In [ ]:
from transformers.adapters import LoRAConfig

config = LoRAConfig(r=8, alpha=16)
model.add_adapter("lora_adapter", config=config)

В статье [Hu et al. (2021)](https://arxiv.org/pdf/2106.09685.pdf) также уделяют особое внимание тому, чтобы свести к минимуму время обучения по сравнению с полным дообучением. Для этого репараметризация LoRA может быть объединена с исходными предварительно обученными весами модели. Таким образом, веса адаптера напрямую используются в каждом forward pass без передачи активаций через дополнительный модуль. В `adapter-transformers`
 это можно реализовать с помощью встроенного метода `merge_adapter()`:

In [ ]:
model.merge_adapter("lora_adapter")

Чтобы продолжить обучение с этим адаптером LoRA или полностью отключить его, сначала необходимо снова сбросить объединенные веса:

In [ ]:
model.reset_adapter("lora_adapter")

**Материалы:**

1. https://arxiv.org/pdf/2106.09685.pdf
2. https://adapterhub.ml/blog/2022/09/updates-in-adapter-transformers-v3-1/

## `3.8 Измерение качества, 🤗Evaluate`

`Trainer` не оценивает автоматически производительность модели во время обучения. Вам нужно будет передать `Trainer` функцию для вычисления метрик и отчета о них. Библиотека 🤗Evaluate предоставляет набор простых функций, которые вы можете загрузить с помощью функции `evaluate.load`.

In [ ]:
import numpy as np
import evaluate

acc_score = evaluate.load("accuracy")
f1_score = evaluate.load("f1")
seq_score = evaluate.load("seqeval")

Вызовите `compute` для `metric`, чтобы рассчитать измерить качество ваших прогнозов. Прежде чем передавать свои прогнозы для вычисления, вам необходимо преобразовать прогнозы в логиты (помните, что все модели 🤗Transformers возвращают логиты):

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return f1_score.compute(predictions=predictions, references=labels, average='micro')

Если вы хотите отслеживать метрики во время fine-tuning, укажите параметр `evaluation_strategy` в аргументах тренера, чтобы выводить метрику оценки в конце каждой эпохи:
```python
training_args = TrainingArguments(...output_dir="test_trainer", evaluation_strategy="epoch"...)
trainer = Trainer(...compute_metrics=compute_metrics...)
```

# `3. GPT`

Исторически, GPT вышел раньше BERT-а, в 2018 году. Сама по себе архитектура является просто стеком декодеров из трансформера:

<center>
<div>
<img src="https://jalammar.github.io/images/gpt2/gpt2-sizes-hyperparameters-3.png", width="700">
</div>
</center>

Довольно занимательным является факт, что с момента выхода GPT, архитектура этой модели особо не изменялась, увеличивалось только количество параметров. К примеру, в GPT-3 было уже 96 декодеров в стеке
.

GPT учится предсказывать следующий токен по левому контексту. Для этих целей идеально подошел Transformer-Decoder. Чтобы GPT не мог подглядывать в правую сторону используется маскирование.
<center>
<div>
<img src="https://jalammar.github.io/images/xlnet/transformer-decoder-block-self-attention-2.png", width="700">
</div>
</center>


**Beyond Language Modeling**

Оказалось, что сам по себе Decoder трансформера является довольно самодостаточной архитектурой, способной обучаться на seq-to-seq задачи и справляться с ними не хуже оригинальной архитектуры Encoder-Decoder

Решение задачи машинного перевода может выглядеть так:
<center>
<div>
<img src="https://jalammar.github.io/images/gpt2/decoder-only-transformer-translation.png", width="700">
</div>

Аналогичным образом можно решать задачи суммаризации текста:
<center>
<div>
<img src="https://jalammar.github.io/images/gpt2/decoder-only-summarization.png", width="700">
</div>

Однако, на практике мы будем чаще сталкиваться с задачами, которые лучше решаются c помощью BERT

**Материалы**
1. https://jalammar.github.io/illustrated-gpt2/
2. https://www.youtube.com/watch?v=qGkzHFllWDY&list=PLoROMvodv4rNiJRchCzutFw5ItR_Z27CM

## `3.1 Text generation`

Давайте решим задачу генерации текста с помощью 🤗Transformers и gpt2.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [ ]:
def load_tokenizer_and_model(model_name_or_path):
  return GPT2Tokenizer.from_pretrained(model_name_or_path), GPT2LMHeadModel.from_pretrained(model_name_or_path).cuda()


def generate(
    model, tok, text,
    do_sample=True, max_length=50, repetition_penalty=5.0,
    top_k=5, top_p=0.95, temperature=1,
    num_beams=None,
    no_repeat_ngram_size=3
    ):
  input_ids = tok.encode(text, return_tensors="pt").cuda()
  out = model.generate(
      input_ids.cuda(),
      max_length=max_length,
      repetition_penalty=repetition_penalty,
      do_sample=do_sample,
      top_k=top_k, top_p=top_p, temperature=temperature,
      num_beams=num_beams, no_repeat_ngram_size=no_repeat_ngram_size
      )
  return list(map(tok.decode, out))

**RuGPT3Large**

In [ ]:
# sber renamed themselves in hub
# tok, model = load_tokenizer_and_model("sberbank-ai/rugpt3large_based_on_gpt2")
tok, model = load_tokenizer_and_model("ai-forever/rugpt3large_based_on_gpt2")
generated = generate(model, tok, "Сегодня утром у меня было прекрасное настроение потому что", num_beams=10, max_length=500)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [ ]:
generated[0]

'Сегодня утром у меня было прекрасное настроение потому что я наконец-то нашла себе работу. Я работаю в компании, которая занимается разработкой и внедрением программного обеспечения на базе 1С:Бухгалтерия 8 (1С:Предприятие 8), а также консультированием пользователей по работе с программным обеспечением фирмы "1С". \n В этой статье я хочу рассказать вам о том, как мне удалось устроиться на работу в одну из самых крупных компаний нашей страны – ООО "ИнфоТехнолоджис", занимающуюся поставками оборудования для нефтегазовой отрасли. Эта компания является одним из крупнейших поставщиков нефтегазового оборудования во всем мире. На сегодняшний день она поставляет свою продукцию более чем в 100 стран мира. И это далеко не полный список тех стран, которые сотрудничают с данной компанией. Но обо всем по порядку. \n Здравствуйте, уважаемые читатели блога KtoNaNovenkogo.ru! Сегодня мы поговорим об одном очень интересном способе заработка в интернете без вложений. Речь пойдет о так называемом пассив

In [ ]:
generated = generate(model, tok, "Парки Москвы очень преобразились за последние 10 лет", num_beams=10, max_length=500)
generated[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


'Парки Москвы очень преобразились за последние 10 лет. В них появилось много новых аттракционов, в том числе и для детей.\nНапример, на ВДНХ открылся парк развлечений «Кванториум», где можно покататься на квадроциклах, поиграть в пинг-понг, а также прокатиться на роликовых коньках или скейтбордах. Кроме того, посетители парка смогут принять участие в мастер-классах по изготовлению различных поделок из природных материалов: глины, папье-маше, бисера, бусин, ракушек и т.д.\nВ парке «Сокольники» появилась новая интерактивная площадка под названием «Парк динозавров». Здесь посетителям предложат поучаствовать в увлекательной викторине с разгадыванием загадок об этом удивительном животном. Также здесь можно будет посмотреть мультфильм о приключениях одного из самых опасных хищников планеты – тираннозавра Рекса.\nКроме того, во многих парках столицы появились новые аттракционы — карусели, горки, веревочные лестницы, батуты, качели и многое другое.<s>\nУзнай как замшелые убеждения, стереотипы,

In [ ]:
generated = generate(model, tok, "Через 100 лет человечество будет способно", num_beams=10, max_length=250)
generated[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


'Через 100 лет человечество будет способно создать искусственный интеллект, который сможет не только читать мысли человека, но и предугадывать его действия. Об этом говорится в статье, опубликованной в научном журнале Proceedings of the National Academy of Sciences (PNAS).\nИсследователи из Стэнфордского института науки и технологий (Stanford Institute for Science and Technology) опубликовали статью о том, что к 2050 году люди будут способны создавать искусственные нейронные сети для того, чтобы предсказывать поведение других людей. По мнению исследователей, это станет возможным благодаря тому, что человеческий мозг способен обрабатывать огромное количество информации за короткий промежуток времени. При этом ученые отмечают, что создание искусственных нейронных сетей может быть сопряжено с определенными трудностями. В частности, речь идет об использовании большого количества вычислительных мощностей. Кроме того, исследователи указывают на то, что при создании искусственного интеллекта 

**Ссылки**

1. https://github.com/ai-forever/ru-gpts